# RANGE Evaluation of Published SatCLIP Models

This notebook evaluates the **published SatCLIP models** (L=10 and L=40) on the RANGE benchmark tasks.

**RANGE Tasks:**
- **Classification:** biome, ecoregion, country, ocean/land
- **Regression:** temperature, housing, elevation, population
- **Synthetic:** checkerboard patterns

**Methodology:** Extract embeddings from location encoder, fit Ridge classifier/regressor, measure accuracy/R².

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_REPO/blob/main/notebooks/RANGE_Eval_Published_SatCLIP.ipynb)

## 1. Setup Environment

In [ ]:
# Clone the original SatCLIP repository
!rm -rf satclip_repo
!git clone https://github.com/microsoft/satclip.git satclip_repo

In [ ]:
# Install dependencies
!pip install lightning --quiet
!pip install torchgeo --quiet
!pip install huggingface_hub --quiet
!pip install einops --quiet

In [ ]:
import sys
sys.path.append('./satclip_repo/satclip')

import os
import gc
import zipfile
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from tqdm.notebook import tqdm
from sklearn.linear_model import RidgeCV, RidgeClassifierCV
from sklearn.preprocessing import MinMaxScaler
from huggingface_hub import hf_hub_download

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load RANGE Evaluation Data from Google Drive

The evaluation datasets include:
- Ecoregion/Biome classification
- Country classification  
- Land/Ocean classification
- Temperature regression
- Housing price regression (California)
- Elevation regression
- Population regression

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Extract RANGE evaluation data from Google Drive
DATA_DIR = './range_eval_data'
os.makedirs(DATA_DIR, exist_ok=True)

print("Extracting RANGE evaluation data...")
with zipfile.ZipFile('/content/drive/MyDrive/grad/learned_activations/range_eval_data.zip', 'r') as z:
    z.extractall(DATA_DIR)

# The zip extracts to range_eval_data/range_eval_data/, so update path if needed
if os.path.exists(os.path.join(DATA_DIR, 'range_eval_data')):
    DATA_DIR = os.path.join(DATA_DIR, 'range_eval_data')

print(f"\nData directory: {DATA_DIR}")
print(f"Files found:")
for f in sorted(os.listdir(DATA_DIR)):
    print(f"  - {f}")

## 3. Load Published SatCLIP Models

In [ ]:
from load import get_satclip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Download and load SatCLIP L=10 (low resolution)
print("\nLoading SatCLIP ResNet18 L=10...")
satclip_l10_path = hf_hub_download("microsoft/SatCLIP-ResNet18-L10", "satclip-resnet18-l10.ckpt")
satclip_l10 = get_satclip(satclip_l10_path, device=device)
satclip_l10.eval()
print(f"  Loaded!")

# Download and load SatCLIP L=40 (high resolution)
print("\nLoading SatCLIP ResNet18 L=40...")
satclip_l40_path = hf_hub_download("microsoft/SatCLIP-ResNet18-L40", "satclip-resnet18-l40.ckpt")
satclip_l40 = get_satclip(satclip_l40_path, device=device)
satclip_l40.eval()
print(f"  Loaded!")

In [ ]:
# Quick test - verify models work
test_coords = torch.randn(5, 2).to(device)  # lon, lat

with torch.no_grad():
    emb_l10 = satclip_l10(test_coords.double())
    emb_l40 = satclip_l40(test_coords.double())

print(f"L=10 output shape: {emb_l10.shape}")
print(f"L=40 output shape: {emb_l40.shape}")

## 4. RANGE Evaluation Code

In [ ]:
# Configuration (consistent with original RANGE evaluation)
RANDOM_SEED = 42
RIDGE_ALPHAS = (0.1, 1.0, 10.0)
TRAIN_RATIO = 0.8
VAL_RATIO = 0.2

In [ ]:
# Dataset Classes

class BiomeDataset(Dataset):
    """Biome classification dataset from ecoregion data."""
    def __init__(self, data_dir):
        train_df = pd.read_csv(os.path.join(data_dir, 'ecoregion_train.csv'))
        val_df = pd.read_csv(os.path.join(data_dir, 'ecoregion_val.csv'))
        df = pd.concat([train_df, val_df])
        df = df.dropna(subset=['BIOME_NAME']).reset_index(drop=True)
        self.labels, self.label_map = pd.factorize(df['BIOME_NAME'])
        self.locations = df[['X', 'Y']].values
        self.num_classes = df['BIOME_NAME'].nunique()

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return torch.from_numpy(self.locations[idx]).double(), self.labels[idx]


class EcoregionDataset(Dataset):
    """Ecoregion classification dataset."""
    def __init__(self, data_dir):
        train_df = pd.read_csv(os.path.join(data_dir, 'ecoregion_train.csv'))
        val_df = pd.read_csv(os.path.join(data_dir, 'ecoregion_val.csv'))
        df = pd.concat([train_df, val_df])
        df = df.dropna(subset=['ECO_NAME']).reset_index(drop=True)
        self.labels, self.label_map = pd.factorize(df['ECO_NAME'])
        self.locations = df[['X', 'Y']].values
        self.num_classes = df['ECO_NAME'].nunique()

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return torch.from_numpy(self.locations[idx]).double(), self.labels[idx]


class CountryDataset(Dataset):
    """Country classification dataset."""
    def __init__(self, data_path):
        df = pd.read_csv(data_path)
        df = df.dropna(subset=['country', 'lat', 'lon']).reset_index(drop=True)
        self.labels, self.label_map = pd.factorize(df['country'])
        self.locations = df[['lon', 'lat']].values
        self.num_classes = df['country'].nunique()

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return torch.from_numpy(self.locations[idx]).double(), self.labels[idx]


class OceanDataset(Dataset):
    """Ocean/land binary classification dataset."""
    def __init__(self, data_path):
        df = pd.read_csv(data_path)
        df = df.dropna(subset=['land', 'lat', 'lon']).reset_index(drop=True)
        self.labels = df['land'].values
        self.locations = df[['lon', 'lat']].values
        self.num_classes = df['land'].nunique()

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return torch.from_numpy(self.locations[idx]).double(), self.labels[idx]


class TemperatureDataset(Dataset):
    """Mean temperature regression dataset."""
    def __init__(self, data_path):
        df = pd.read_csv(data_path)
        df = df.dropna(subset=['meanT']).reset_index(drop=True)
        self.labels = df['meanT'].values
        self.locations = df[['Lon', 'Lat']].values
        self.num_classes = 0

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return torch.from_numpy(self.locations[idx]).double(), torch.tensor(self.labels[idx]).double()


class HousingDataset(Dataset):
    """California housing prices regression dataset."""
    def __init__(self, data_path):
        df = pd.read_csv(data_path)
        df = df.dropna(subset=['median_house_value']).reset_index(drop=True)
        self.labels = df['median_house_value'].values
        self.locations = df[['longitude', 'latitude']].values
        self.num_classes = 0

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return torch.from_numpy(self.locations[idx]).double(), torch.tensor(self.labels[idx]).double()


class ElevationDataset(Dataset):
    """Elevation regression dataset."""
    def __init__(self, data_path):
        df = pd.read_csv(data_path)
        df = df.dropna(subset=['elevation']).reset_index(drop=True)
        self.labels = df['elevation'].values
        self.locations = df[['lon', 'lat']].values
        self.num_classes = 0

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return torch.from_numpy(self.locations[idx]).double(), torch.tensor(self.labels[idx]).double()


class PopulationDataset(Dataset):
    """Population regression dataset (log-transformed)."""
    def __init__(self, data_path):
        df = pd.read_csv(data_path)
        df = df.dropna(subset=['population']).reset_index(drop=True)
        self.labels = df['population'].values
        self.locations = df[['lon', 'lat']].values
        self.num_classes = 0

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        loc = torch.from_numpy(self.locations[idx]).double()
        label = np.log(1 + self.labels[idx])  # Log transform
        return loc, torch.tensor(label).double()

In [ ]:
# Checkerboard Dataset (Synthetic)

def generate_fibonacci_lattice(n_points, n_classes=16):
    """Generate points on a sphere using Fibonacci lattice sampling."""
    import math
    n_points = n_points // 2
    phi = (1 + math.sqrt(5)) / 2
    lats, lons, labels = [], [], []
    for i in np.arange(-n_points, n_points):
        lat = np.arcsin((2 * i) / (2 * n_points + 1)) * 180 / np.pi
        lon = (i % phi) * (360 / phi)
        if lon < -180: lon += 360
        if lon > 180: lon -= 360
        lons.append(lon)
        lats.append(lat)
        labels.append(i % n_classes)
    return np.array(lons), np.array(lats), np.array(labels)


def haversine_distance(lon1, lat1, lon2, lat2, radius=1.0):
    """Calculate pairwise Haversine distances."""
    lon1, lat1 = np.radians(lon1), np.radians(lat1)
    lon2, lat2 = np.radians(lon2), np.radians(lat2)
    dlon = lon2[:, np.newaxis] - lon1
    dlat = lat2[:, np.newaxis] - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2[:, np.newaxis]) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return radius * c


def cart2sph(x, y, z):
    hxy = np.hypot(x, y)
    el = np.arctan2(z, hxy)
    az = np.arctan2(y, x)
    return az, el


def get_checker_data(n_samples, n_support, n_classes, seed=0, grid=False):
    lons, lats, labels = generate_fibonacci_lattice(n_support, n_classes=n_classes)
    if grid:
        lons_grid, lats_grid, _ = generate_fibonacci_lattice(n_samples)
        distances = haversine_distance(lons_grid, lats_grid, lons, lats)
        labels_grid = labels[distances.argmin(0)]
        lonlats = torch.from_numpy(np.stack([lons_grid, lats_grid]).T.astype(np.float64))
        labels_out = torch.from_numpy(labels_grid.astype(np.int64))
    else:
        rng = np.random.RandomState(seed)
        x, y, z = rng.normal(size=(3, n_samples))
        az, el = cart2sph(x, y, z)
        lons_seed, lats_seed = np.rad2deg(az), np.rad2deg(el)
        distances = haversine_distance(lons_seed, lats_seed, lons, lats)
        labels_seed = labels[distances.argmin(0)]
        lonlats = torch.from_numpy(np.stack([lons_seed, lats_seed]).T.astype(np.float64))
        labels_out = torch.from_numpy(labels_seed.astype(np.int64))
    return lonlats, labels_out


class CheckerboardDataset:
    def __init__(self, n_samples=10000, n_classes=16, n_support=200):
        self.train_ds = TensorDataset(*get_checker_data(n_samples, n_support, n_classes, seed=0))
        self.valid_ds = TensorDataset(*get_checker_data(n_samples, n_support, n_classes, seed=1))
        self.eval_ds = TensorDataset(*get_checker_data(n_samples, n_support, n_classes, grid=True))

In [ ]:
# Dataset Loading

def get_dataset(task_name, data_dir, batch_size=256, num_workers=0):
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    
    if task_name == 'biome':
        dataset = BiomeDataset(data_dir)
        train_ds, val_ds = random_split(dataset, [TRAIN_RATIO, VAL_RATIO], generator=generator)
        num_classes = dataset.num_classes
        task_type = 'classification'
    elif task_name == 'ecoregion':
        dataset = EcoregionDataset(data_dir)
        train_ds, val_ds = random_split(dataset, [TRAIN_RATIO, VAL_RATIO], generator=generator)
        num_classes = dataset.num_classes
        task_type = 'classification'
    elif task_name == 'country':
        dataset = CountryDataset(os.path.join(data_dir, 'country.csv'))
        train_ds, val_ds = random_split(dataset, [TRAIN_RATIO, VAL_RATIO], generator=generator)
        num_classes = dataset.num_classes
        task_type = 'classification'
    elif task_name == 'ocean':
        train_ds = OceanDataset(os.path.join(data_dir, 'land_ocean_train.csv'))
        val_ds = OceanDataset(os.path.join(data_dir, 'land_ocean_test.csv'))
        num_classes = train_ds.num_classes
        task_type = 'classification'
    elif task_name == 'temperature':
        dataset = TemperatureDataset(os.path.join(data_dir, 'temp.csv'))
        train_ds, val_ds = random_split(dataset, [TRAIN_RATIO, VAL_RATIO], generator=generator)
        num_classes = 0
        task_type = 'regression'
    elif task_name == 'housing':
        dataset = HousingDataset(os.path.join(data_dir, 'housing.csv'))
        train_ds, val_ds = random_split(dataset, [TRAIN_RATIO, VAL_RATIO], generator=generator)
        num_classes = 0
        task_type = 'regression'
    elif task_name == 'elevation':
        dataset = ElevationDataset(os.path.join(data_dir, 'elevation.csv'))
        train_ds, val_ds = random_split(dataset, [TRAIN_RATIO, VAL_RATIO], generator=generator)
        num_classes = 0
        task_type = 'regression'
    elif task_name == 'population':
        dataset = PopulationDataset(os.path.join(data_dir, 'population.csv'))
        train_ds, val_ds = random_split(dataset, [TRAIN_RATIO, VAL_RATIO], generator=generator)
        num_classes = 0
        task_type = 'regression'
    elif task_name.startswith('checker_'):
        n_support = int(task_name.split('_')[1])
        checker = CheckerboardDataset(n_samples=10000, n_classes=16, n_support=n_support)
        train_ds = checker.train_ds
        val_ds = checker.eval_ds
        num_classes = 16
        task_type = 'classification'
    else:
        raise ValueError(f"Unknown task: {task_name}")
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=num_workers, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers, shuffle=False)
    
    return train_loader, val_loader, num_classes, task_type

In [ ]:
# Embedding Extraction and Evaluation

def extract_embeddings(data_loader, model, device='cuda'):
    """Extract embeddings from a location encoder model."""
    model.eval()
    embeddings_list = []
    labels_list = []
    
    with torch.no_grad():
        for coords, labels in tqdm(data_loader, desc="Extracting embeddings", leave=False):
            coords = coords.double().to(device)
            emb = model(coords).cpu().numpy()
            embeddings_list.append(emb)
            labels_list.append(labels.numpy() if isinstance(labels, torch.Tensor) else labels)
    
    return np.concatenate(embeddings_list), np.concatenate(labels_list)


def evaluate_embeddings(train_emb, train_labels, val_emb, val_labels, task_type):
    """Evaluate embeddings using Ridge regression/classification."""
    scaler = MinMaxScaler()
    train_emb = scaler.fit_transform(train_emb)
    val_emb = scaler.transform(val_emb)
    
    if task_type == 'classification':
        model = RidgeClassifierCV(alphas=RIDGE_ALPHAS, cv=10)
    else:
        model = RidgeCV(alphas=RIDGE_ALPHAS, cv=3)
    
    model.fit(train_emb, train_labels)
    return model.score(val_emb, val_labels)


def evaluate_task(task_name, model, data_dir, device='cuda', batch_size=256):
    """Full evaluation pipeline for a single task."""
    train_loader, val_loader, num_classes, task_type = get_dataset(task_name, data_dir, batch_size)
    
    train_emb, train_labels = extract_embeddings(train_loader, model, device)
    val_emb, val_labels = extract_embeddings(val_loader, model, device)
    
    score = evaluate_embeddings(train_emb, train_labels, val_emb, val_labels, task_type)
    
    metric = "Accuracy" if task_type == 'classification' else "R²"
    print(f"  {task_name:15s}: {score:.4f} ({metric})")
    
    return score


def evaluate_all_tasks(model, model_name, data_dir, device='cuda', tasks=None):
    """Evaluate model on all tasks."""
    if tasks is None:
        tasks = [
            'biome', 'ecoregion', 'country', 'ocean',
            'temperature', 'housing', 'elevation', 'population',
            'checker_100', 'checker_200',
        ]
    
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    results = {}
    for task in tasks:
        try:
            score = evaluate_task(task, model, data_dir, device)
            results[task] = score
        except Exception as e:
            print(f"  {task:15s}: FAILED ({e})")
            results[task] = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return results

## 5. Run RANGE Evaluation

In [ ]:
# Evaluate SatCLIP L=10
results_l10 = evaluate_all_tasks(satclip_l10, "SatCLIP ResNet18 L=10", DATA_DIR, device)

In [ ]:
# Evaluate SatCLIP L=40
results_l40 = evaluate_all_tasks(satclip_l40, "SatCLIP ResNet18 L=40", DATA_DIR, device)

## 6. Results Summary

In [ ]:
# Create results comparison table
print("\n" + "="*70)
print("RANGE EVALUATION RESULTS - Published SatCLIP Models")
print("="*70)
print(f"{'Task':<20} {'L=10':<15} {'L=40':<15} {'Metric'}")
print("-"*70)

classification_tasks = ['biome', 'ecoregion', 'country', 'ocean', 'checker_100', 'checker_200']
regression_tasks = ['temperature', 'housing', 'elevation', 'population']

print("\nClassification Tasks (Accuracy):")
for task in classification_tasks:
    l10 = results_l10.get(task)
    l40 = results_l40.get(task)
    l10_str = f"{l10:.4f}" if l10 is not None else "FAILED"
    l40_str = f"{l40:.4f}" if l40 is not None else "FAILED"
    print(f"  {task:<18} {l10_str:<15} {l40_str:<15}")

print("\nRegression Tasks (R²):")
for task in regression_tasks:
    l10 = results_l10.get(task)
    l40 = results_l40.get(task)
    l10_str = f"{l10:.4f}" if l10 is not None else "FAILED"
    l40_str = f"{l40:.4f}" if l40 is not None else "FAILED"
    print(f"  {task:<18} {l10_str:<15} {l40_str:<15}")

print("\n" + "="*70)

In [ ]:
# Create a pandas DataFrame for easy export
results_df = pd.DataFrame({
    'Task': list(results_l10.keys()),
    'SatCLIP_L10': list(results_l10.values()),
    'SatCLIP_L40': list(results_l40.values()),
})

# Add task type
results_df['Type'] = results_df['Task'].apply(
    lambda x: 'Classification' if x in classification_tasks else 'Regression'
)

display(results_df)

In [ ]:
# Save results to CSV
results_df.to_csv('satclip_range_results.csv', index=False)
print("Results saved to satclip_range_results.csv")

# Also save to Google Drive for persistence
results_df.to_csv('/content/drive/MyDrive/grad/learned_activations/satclip_range_results.csv', index=False)
print("Also saved to Google Drive!")

## 7. Comparison with Your Models (Optional)

If you want to compare with your trained models, add their results here:

In [ ]:
# Your model results from the HPC evaluation (from conversation context)
your_results = {
    'Spline+SH': {
        'biome': 0.7640, 'ecoregion': 0.6720, 'country': 0.9234, 'ocean': 0.9590,
        'temperature': 0.8986, 'housing': 0.5705, 'elevation': 0.7341, 'population': 0.7541,
        'checker_100': 0.9099, 'checker_200': 0.8537
    },
    'SIREN+SH': {
        'biome': 0.7632, 'ecoregion': 0.6409, 'country': 0.9301, 'ocean': 0.9606,
        'temperature': 0.9142, 'housing': 0.3775, 'elevation': 0.7694, 'population': 0.7777,
        'checker_100': 0.9225, 'checker_200': 0.8719
    },
}

# Add published SatCLIP results
your_results['SatCLIP_L10'] = results_l10
your_results['SatCLIP_L40'] = results_l40

# Create comparison DataFrame
comparison_df = pd.DataFrame(your_results)
comparison_df.index.name = 'Task'
display(comparison_df.round(4))